In [133]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [134]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Read Data

In [135]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
gendered1 = pd.read_csv('../data/letters_2021_processed_with_gender.csv')
gendered1 = gendered1[['s1_s2', 'LETTER_GENDER']]
gendered1 = gendered1.rename(columns={'LETTER_GENDER':'label'})

In [136]:
gendered1[gendered1['GENDERLESS_TEXT'].str.contains('julie', case=False, na=False)]['GENDERLESS_TEXT']

KeyError: 'GENDERLESS_TEXT'

In [ ]:
gendered2 = pd.read_csv('../data/sentence_sets_trimmed_processed_with_gender.csv', encoding='mac-roman')
gendered2 = gendered2[['s1_s2', 'applicant_gender']]
gendered2 = gendered2.rename(columns={'TEXT':'LETTERTEXT', 'applicant_gender':'label'})

In [ ]:
df = pd.concat([gendered1, gendered2], ignore_index=True)

In [ ]:
gender_label_mapping = {
    'F':0,
    'female':0,
    'M':1,
    'male':1
}

In [ ]:
df['label'] = df['label'].replace(gender_label_mapping)

# Get Unique Words

In [137]:
def tokenize(text):

    tokens = re.findall(r'\b\w+\b', text.lower())
    return set(tokens)

df['tokens'] = df['s1_s2'].apply(tokenize)

In [138]:
male_tokens = set().union(*df[df['label'] == 1]['tokens'])
female_tokens = set().union(*df[df['label'] == 0]['tokens'])

In [139]:
unique_to_male = male_tokens - female_tokens
unique_to_female = female_tokens - male_tokens

In [140]:
unique_to_male

{'northstate',
 'blemish',
 'hellenic',
 'caters',
 '7msrn8zef2',
 'adores',
 'monster',
 'souweidane',
 'amd',
 'chronid',
 'television',
 'lpn',
 'charger',
 'attributing',
 'inferior',
 'particulars',
 'ade',
 'ppanan',
 'safeguarding',
 'thathave',
 'bacteriophage',
 'recognise',
 'slideshows',
 'hum',
 'acetic',
 'counselors',
 'benignant',
 'discretely',
 'bayview',
 'neurofibromastosis',
 'warfare',
 'bse',
 'bang',
 'recruiter',
 'islamabad',
 'machu',
 'dupont',
 'intermural',
 'desaturating',
 'carving',
 'cellphone',
 'converted',
 'unorthodox',
 'dosa',
 'gmos',
 'fthe',
 'granular',
 'tibor',
 '7n63t4mec',
 'wrnmmc',
 'glycocalyx',
 'housetaff',
 'baja',
 'gratis',
 'crashing',
 'laker',
 'fount',
 'clicking',
 'goodhearted',
 'correlating',
 'wilin',
 'fearful',
 'watson',
 'lease',
 'ignite',
 'haemodynamic',
 'saxa',
 'dispense',
 'otheranesthesiologists',
 'gurneys',
 'mamoona',
 'krasnow',
 'jealous',
 'pique',
 'intercommunication',
 'pathologically',
 'appreciable',

In [141]:
unique_to_female

{'inf',
 'joyous',
 'hydrops',
 'amboss',
 'epa',
 'broached',
 'sexfw',
 'myopathy',
 'dysautonomia',
 'disengaged',
 'corresponds',
 'vpma',
 'imprecise',
 'muslims',
 'guilty',
 'univeristy',
 'gracefulness',
 'thg',
 'isoform',
 'shewas',
 'recuperating',
 'terwards',
 'ren',
 'mandated',
 'severance',
 'glycopyrrolate',
 'vicsom',
 'm54',
 'pharynx',
 'bridget',
 'radiography',
 'lookup',
 'adulthood',
 'textiles',
 'amend',
 'hyperalgesia',
 'pierce',
 'shade',
 'fhm',
 'watchman',
 'smoothing',
 'beverly',
 'reinnervation',
 'sonogram',
 'pens',
 'overachievement',
 'broomfield',
 'requisition',
 'poetry',
 'sarnoff',
 'ghrelin',
 'modifier',
 'oliva',
 'captivate',
 'zumba',
 't8',
 'cpcr',
 'commutes',
 'ornaments',
 '5067',
 'anest',
 'plugging',
 'modrall',
 'lengthier',
 'steel',
 'midday',
 'adnexal',
 'inseparably',
 'honorific',
 'glargine',
 'anumber',
 'chatted',
 'unexceptionally',
 'provocation',
 'cardioplegia',
 'anonymous',
 'f1000',
 'n3',
 'atony',
 'schreiner',

# TF-IDF

In [142]:

male_texts = df[df['label'] == 1]['s1_s2']
female_texts = df[df['label'] == 0]['s1_s2']

In [143]:
vectorizer = TfidfVectorizer(lowercase=True, stop_words='english', max_features=5000)

In [144]:
male_tfidf = vectorizer.fit_transform(male_texts)
male_vocab = vectorizer.get_feature_names_out()
male_scores = male_tfidf.mean(axis=0).A1

female_tfidf = vectorizer.fit_transform(female_texts)
female_vocab = vectorizer.get_feature_names_out()
female_scores = female_tfidf.mean(axis=0).A1


In [145]:
male_df = pd.DataFrame({'token': male_vocab, 'male_score': male_scores})
female_df = pd.DataFrame({'token': female_vocab, 'female_score': female_scores})

tfidf_df = pd.merge(male_df, female_df, on='token', how='outer').fillna(0)

tfidf_df['score_diff'] = tfidf_df['male_score'] - tfidf_df['female_score']

top_male_tokens = tfidf_df.sort_values(by='score_diff', ascending=False).head(20)
top_female_tokens = tfidf_df.sort_values(by='score_diff').head(20)


In [146]:
top_male_tokens

,token,male_score,female_score,score_diff
3304,mr,0.026335,0.000295,0.026039
2045,first_name,0.105659,0.087074,0.018586
3073,man,0.005402,0.000330,0.005072
5598,äö,0.021737,0.019430,0.002307
723,calm,0.007770,0.005907,0.001864
4772,staff,0.017290,0.015465,0.001825
5588,young,0.007450,0.005746,0.001704
4617,showed,0.012498,0.010834,0.001664
3768,physician,0.015756,0.014116,0.001640
2955,liked,0.007661,0.006214,0.001447


In [147]:
top_female_tokens

,token,male_score,female_score,score_diff
3307,ms,0.002079,0.028230,-0.026151
2478,identifier,0.154875,0.160422,-0.005546
2344,health,0.010667,0.014453,-0.003786
5544,woman,0.000348,0.003688,-0.003340
3221,middle_name,0.015395,0.018397,-0.003002
4319,research,0.024801,0.027745,-0.002945
5545,women,0.000821,0.003224,-0.002403
4955,surgery,0.015753,0.017974,-0.002222
3587,outstanding,0.016482,0.018648,-0.002165
753,care,0.030758,0.032505,-0.001747


# TF-IDF on Two Male and Female Documents

In [148]:
male_doc = ' '.join(df[df['label'] == 1]['s1_s2'].tolist())
female_doc = ' '.join(df[df['label'] == 0]['s1_s2'].tolist())

corpus = [male_doc, female_doc]
labels = ['male', 'female']

In [149]:
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform([male_doc, female_doc])
tokens = vectorizer.get_feature_names_out()

counts = pd.DataFrame(X.toarray().T, index=tokens, columns=['male', 'female'])

counts += 1e-5

counts['log_ratio'] = np.log(counts['male'] / counts['female'])

top_male = counts.sort_values(by='log_ratio', ascending=False).head(18)
top_female = counts.sort_values(by='log_ratio').head(18)

print("Top male-associated tokens:")
print(top_male)

print("\nTop female-associated tokens:")
print(top_female)

Top male-associated tokens:
                male   female  log_ratio
squadron    38.00001  0.00001  15.150512
guy         35.00001  0.00001  15.068274
scout       32.00001  0.00001  14.978662
eagle       27.00001  0.00001  14.808763
dylan       26.00001  0.00001  14.771022
2d          26.00001  0.00001  14.771022
jin         24.00001  0.00001  14.690980
lcdr        22.00001  0.00001  14.603968
undersea    20.00001  0.00001  14.508658
yong        20.00001  0.00001  14.508658
gas         19.00001  0.00001  14.457365
ordering    16.00001  0.00001  14.285515
reese       16.00001  0.00001  14.285515
aviation    16.00001  0.00001  14.285515
huntington  16.00001  0.00001  14.285515
outdoor     15.00001  0.00001  14.220976
saving      15.00001  0.00001  14.220976
chung       15.00001  0.00001  14.220976

Top female-associated tokens:
            male    female  log_ratio
julie    0.00001  32.00001 -14.978662
mrs      0.00001  27.00001 -14.808763
alice    0.00001  25.00001 -14.731802
ding     0